In [1]:
from pathlib import Path
import re
import pandas as pd

RAVDESS_ROOT = Path(r"C:\Code\Python\data\RAVDESS")
CREMAD_ROOT  = Path(r"C:\Code\Python\data\CREMA-D\AudioWAV")
IEMOCAP_ROOT = Path(r"C:\Code\Python\data\IEMOCAP\IEMOCAP_full_release")


In [2]:
ravdess_files = sorted(RAVDESS_ROOT.glob("**/*.wav"))
print("RAVDESS files:", len(ravdess_files))

emotion_map_ravdess = {
    "01": "neutral",
    "02": "neutral",  # calm merged with neutral
    "03": "happy",
    "04": "sad",
    "05": "angry",
}

def parse_ravdess(filepath: Path):
    stem = filepath.stem  # "03-01-01-01-01-01-01"
    parts = stem.split("-")
    if len(parts) != 7:
        return None

    emotion_code = parts[2]
    actor_id = parts[6]

    if emotion_code not in emotion_map_ravdess:
        return None  # skip 06–08

    return {
        "filepath": str(filepath),
        "dataset": "RAVDESS",
        "speaker_id": actor_id,
        "emotion_raw": emotion_code,
        "emotion": emotion_map_ravdess[emotion_code],
    }

ravdess_records = []
for f in ravdess_files:
    rec = parse_ravdess(f)
    if rec is not None:
        ravdess_records.append(rec)

ravdess_df = pd.DataFrame(ravdess_records)
print(ravdess_df.head())
print(ravdess_df["emotion"].value_counts())


RAVDESS files: 2880
                                            filepath  dataset speaker_id  \
0  C:\Code\Python\data\RAVDESS\Actor_01\03-01-01-...  RAVDESS         01   
1  C:\Code\Python\data\RAVDESS\Actor_01\03-01-01-...  RAVDESS         01   
2  C:\Code\Python\data\RAVDESS\Actor_01\03-01-01-...  RAVDESS         01   
3  C:\Code\Python\data\RAVDESS\Actor_01\03-01-01-...  RAVDESS         01   
4  C:\Code\Python\data\RAVDESS\Actor_01\03-01-02-...  RAVDESS         01   

  emotion_raw  emotion  
0          01  neutral  
1          01  neutral  
2          01  neutral  
3          01  neutral  
4          02  neutral  
emotion
neutral    576
happy      384
sad        384
angry      384
Name: count, dtype: int64


In [3]:
cremad_files = sorted(CREMAD_ROOT.glob("*.wav"))
print("CREMA-D files:", len(cremad_files))

emotion_map_cremad = {
    "NEU": "neutral",
    "HAP": "happy",
    "SAD": "sad",
    "ANG": "angry",
}

def parse_cremad(filepath: Path):
    stem = filepath.stem              # "1001_DFA_ANG_XX"
    parts = stem.split("_")
    if len(parts) != 4:
        return None

    speaker_id, sentence, emotion_raw, intensity = parts

    if emotion_raw not in emotion_map_cremad:
        return None  # skip FEA, DIS

    return {
        "filepath": str(filepath),
        "dataset": "CREMA-D",
        "speaker_id": speaker_id,
        "emotion_raw": emotion_raw,
        "emotion": emotion_map_cremad[emotion_raw],
    }

cremad_records = []
for f in cremad_files:
    rec = parse_cremad(f)
    if rec is not None:
        cremad_records.append(rec)

cremad_df = pd.DataFrame(cremad_records)
print(cremad_df.head())
print(cremad_df["emotion"].value_counts())


CREMA-D files: 7442
                                            filepath  dataset speaker_id  \
0  C:\Code\Python\data\CREMA-D\AudioWAV\1001_DFA_...  CREMA-D       1001   
1  C:\Code\Python\data\CREMA-D\AudioWAV\1001_DFA_...  CREMA-D       1001   
2  C:\Code\Python\data\CREMA-D\AudioWAV\1001_DFA_...  CREMA-D       1001   
3  C:\Code\Python\data\CREMA-D\AudioWAV\1001_DFA_...  CREMA-D       1001   
4  C:\Code\Python\data\CREMA-D\AudioWAV\1001_IEO_...  CREMA-D       1001   

  emotion_raw  emotion  
0         ANG    angry  
1         HAP    happy  
2         NEU  neutral  
3         SAD      sad  
4         ANG    angry  
emotion
angry      1271
happy      1271
sad        1271
neutral    1087
Name: count, dtype: int64


In [4]:
def load_iemocap_metadata(root: Path) -> pd.DataFrame:
    allowed_map = {
        "neu": "neutral",
        "hap": "happy",
        "exc": "happy",   # excited merged with happy
        "ang": "angry",
        "sad": "sad",
    }

    records = []

    sessions = [f"Session{i}" for i in range(1, 6)]
    for session in sessions:
        session_dir = root / session

        # 1) Collect all wav files in this session
        wav_paths = list(session_dir.glob("sentences/wav/**/*.wav"))
        wav_index = {p.stem: p for p in wav_paths}

        # 2) Read all EmoEvaluation *.txt files
        label_dir = session_dir / "dialog" / "EmoEvaluation"
        label_files = list(label_dir.glob("*.txt"))

        for lf in label_files:
            with open(lf, "r") as f:
                for line in f:
                    if not line.startswith("["):
                        continue

                    parts = re.split(r"[\t\n]", line.strip())
                    # Expected: [ '[1]', 'Ses01F_impro01_F000', 'ang', '[1.23', '4.56]', ... ]
                    if len(parts) < 3:
                        continue

                    utt_id = parts[1]
                    label_raw = parts[2]

                    if label_raw not in allowed_map:
                        continue
                    if utt_id not in wav_index:
                        continue  # sanity check

                    filepath = wav_index[utt_id]
                    speaker_id = utt_id.split("_")[0]  # e.g. 'Ses01F'

                    records.append({
                        "filepath": str(filepath),
                        "dataset": "IEMOCAP",
                        "speaker_id": speaker_id,
                        "emotion_raw": label_raw,
                        "emotion": allowed_map[label_raw],
                    })

    df = pd.DataFrame(records)
    return df

iemocap_df = load_iemocap_metadata(IEMOCAP_ROOT)
print(iemocap_df.head())
print(iemocap_df["emotion"].value_counts())


                                            filepath  dataset speaker_id  \
0  C:\Code\Python\data\IEMOCAP\IEMOCAP_full_relea...  IEMOCAP     Ses01F   
1  C:\Code\Python\data\IEMOCAP\IEMOCAP_full_relea...  IEMOCAP     Ses01F   
2  C:\Code\Python\data\IEMOCAP\IEMOCAP_full_relea...  IEMOCAP     Ses01F   
3  C:\Code\Python\data\IEMOCAP\IEMOCAP_full_relea...  IEMOCAP     Ses01F   
4  C:\Code\Python\data\IEMOCAP\IEMOCAP_full_relea...  IEMOCAP     Ses01F   

  emotion_raw  emotion  
0         neu  neutral  
1         neu  neutral  
2         neu  neutral  
3         neu  neutral  
4         ang    angry  
emotion
neutral    1708
happy      1636
angry      1103
sad        1084
Name: count, dtype: int64


In [7]:
all_df = pd.concat([ravdess_df, cremad_df, iemocap_df], ignore_index=True)

print("Total utterances:", len(all_df))
print(all_df["dataset"].value_counts())
print(all_df["emotion"].value_counts())
all_df.head()
all_df.to_csv("metadata_step2.csv", index=False)



Total utterances: 12159
dataset
IEMOCAP    5531
CREMA-D    4900
RAVDESS    1728
Name: count, dtype: int64
emotion
neutral    3371
happy      3291
angry      2758
sad        2739
Name: count, dtype: int64
